# Unit 1.1 - Environment Setup & Source Data Exploration

Tiga Aktivitas dan Luaran pada unit 1.1 adalah:
1. Aktivitas Unit 1.1: menyiapkan PostgreSQL dan DBeaver/pgAdmin, memastikan dataset UADW v1.0 dapat diakses, menginventarisasi 10 sumber data, memverifikasi jumlah baris dan kolom, mengenali candidate key, melakukan observasi awal terhadap source, mencatat hubungan awal antar-source, mengisi Computational Thinking Log, serta memastikan raw data tetap read-only.
2. Hasil Unit 1.1: sebuah Source Understanding Package yang berisi Environment Readiness Record, Source Inventory, Import/Access Verification, Initial Exploration Register, Candidate Key Notes, Initial Relationship Notes, CT Log, supporting evidence/query, dan submission checklist. Hasil inilah yang menjadi input langsung untuk Unit 1.2 – Data Exploration & Data Profiling.
3. Dokumentasi Unit 1.1: terutama melalui Template Unit 1.1 Source Inventory Workbook, dengan sheet 01_Environment, 02_Source_Inventory, 03_Import_Verification, 04_Initial_Exploration, 05_Relationship_Notes, 06_CT_Log, dan 07_Checklist. SQL/output query atau screenshot dapat menjadi evidence tambahan.


In [4]:
# Import Library
from pathlib import Path
import pandas as pd

In [8]:
# sesuaikan path dengan lokasi file paket dataset UADW 1.0
BASE_DIR = Path("/Users/adam/Library/CloudStorage/GoogleDrive-adambachtiar@undikma.ac.id/My Drive/PTI-UNDIKMA/Pendidikan/Pengajaran/Data_Warehouse/Dataset/UADW_v1.0/01_raw_data")


SOURCE_META = {
    "program_studi.csv": {"role": "Reference", "candidate_key": "kode_prodi"},
    "semester.csv": {"role": "Reference", "candidate_key": "semester_id"},
    "mahasiswa.csv": {"role": "Master", "candidate_key": "nim"},
    "dosen.csv": {"role": "Master", "candidate_key": "nidn"},
    "mata_kuliah.csv": {"role": "Master/Reference", "candidate_key": "kode_mk"},
    "krs.csv": {"role": "Transaction", "candidate_key": "krs_id"},
    "nilai.csv": {"role": "Transaction", "candidate_key": "nilai_id"},
    "aktivitas_lms.csv": {"role": "Event Transaction", "candidate_key": "lms_event_id"},
    "pembayaran.csv": {"role": "Transaction", "candidate_key": "pembayaran_id"},
    "kelulusan.csv": {"role": "Outcome Transaction", "candidate_key": "kelulusan_id"},
}

In [9]:
inventory = []
for file_name, meta in SOURCE_META.items():
    path = BASE_DIR / file_name
    if not path.exists():
        inventory.append({
            "dataset": file_name,
            "rows": None,
            "columns": None,
            "role": meta["role"],
            "candidate_key": meta["candidate_key"],
            "status": "NOT FOUND",
        })
        continue

    # dtype=str menjaga representasi raw; tidak melakukan type conversion.
    df = pd.read_csv(path, dtype=str, keep_default_na=True)

    inventory.append({
        "dataset": file_name,
        "rows": len(df),
        "columns": len(df.columns),
        "role": meta["role"],
        "candidate_key": meta["candidate_key"],
        "status": "VERIFIED",
    })

inventory_df = pd.DataFrame(inventory)
print(inventory_df.to_string(index=False))

verified_rows = inventory_df["rows"].dropna().sum()
print("\nTotal raw rows:", int(verified_rows))

          dataset   rows  columns                role candidate_key   status
program_studi.csv      6        5           Reference    kode_prodi VERIFIED
     semester.csv      8        6           Reference   semester_id VERIFIED
    mahasiswa.csv   1965       11              Master           nim VERIFIED
        dosen.csv     96        7              Master          nidn VERIFIED
  mata_kuliah.csv    208        7    Master/Reference       kode_mk VERIFIED
          krs.csv  57025        8         Transaction        krs_id VERIFIED
        nilai.csv  57000        8         Transaction      nilai_id VERIFIED
aktivitas_lms.csv 200356        9   Event Transaction  lms_event_id VERIFIED
   pembayaran.csv   9500        8         Transaction pembayaran_id VERIFIED
    kelulusan.csv    347        7 Outcome Transaction  kelulusan_id VERIFIED

Total raw rows: 326511


In [10]:
# ------------------------------------------------------------
# Initial exploration mahasiswa.csv
# ------------------------------------------------------------
mahasiswa_path = BASE_DIR / "mahasiswa.csv"
mahasiswa = pd.read_csv(mahasiswa_path, dtype=str, keep_default_na=True)

raw_rows = len(mahasiswa)
distinct_nim = mahasiswa["nim"].nunique(dropna=True)
excess_duplicate_rows = raw_rows - distinct_nim
missing_kota = (
    mahasiswa["kota_asal_raw"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)
distinct_prodi_raw = mahasiswa["prodi_raw"].nunique(dropna=True)

angkatan_numeric = pd.to_numeric(mahasiswa["angkatan"], errors="coerce")

print("\n=== INITIAL EXPLORATION: mahasiswa.csv ===")
print("Raw rows             :", raw_rows)
print("Distinct NIM         :", distinct_nim)
print("Excess duplicate rows:", excess_duplicate_rows)
print("Missing kota_asal_raw:", int(missing_kota))
print("Distinct prodi_raw   :", distinct_prodi_raw)
print("Minimum angkatan     :", int(angkatan_numeric.min()))
print("Maximum angkatan     :", int(angkatan_numeric.max()))

print("\nDistribusi jk_raw:")
print(mahasiswa["jk_raw"].value_counts(dropna=False).to_string())

print("\nDistribusi status_raw:")
print(mahasiswa["status_raw"].value_counts(dropna=False).to_string())

print("\n=== CATATAN ===")
print("Script ini hanya membaca data.")
print("Tidak ada cleansing, deduplication, UPDATE, atau penulisan ulang raw CSV.")


=== INITIAL EXPLORATION: mahasiswa.csv ===
Raw rows             : 1965
Distinct NIM         : 1950
Excess duplicate rows: 15
Missing kota_asal_raw: 30
Distinct prodi_raw   : 30
Minimum angkatan     : 2021
Maximum angkatan     : 2024

Distribusi jk_raw:
jk_raw
L            764
P            764
Perempuan    183
Laki-laki    162
M             53
F             39

Distribusi status_raw:
status_raw
Aktif     1567
Lulus      348
AKTIF       18
Aktif       16
cuti         9
Cuti         7

=== CATATAN ===
Script ini hanya membaca data.
Tidak ada cleansing, deduplication, UPDATE, atau penulisan ulang raw CSV.
